# CN-PIIBench-Lite — Real Qwen-2.5 Run (Colab GPU)

Two experiments on the paper's **base** Qwen-2.5 weights, 4-bit + batched greedy decoding:

- **Experiment A — Fine-tuning stress test (the headline result).** LoRA-fine-tune the model on synthetic internal records, then audit base vs fine-tuned side by side. Shows induced memorization: base ≈ Low, fine-tuned = High (flagged/rejected).
- **Experiment B (optional) — Base null-floor.** Audit base weights only, across scales; expected ≈ 0 (confirms the benchmark does not false-positive).

> **Before you start:** menu **Runtime → Change runtime type → T4 GPU**, then Save.


## Step 1 — confirm a GPU is attached

In [ ]:
!nvidia-smi

## Step 2 — install dependencies
(~2 min; torch is already on Colab.)

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets pandas

## Step 3 — upload the toolkit
Run the cell, click **Choose Files**, pick **`pii_auditor_pkg.zip`**. You should then see `pii_auditor`, `finetune_stress.py`, `run_gpu.py` listed.

In [ ]:
from google.colab import files
import zipfile, io, os
up = files.upload()
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(io.BytesIO(up[name])).extractall('.')
print('extracted:', sorted(p for p in os.listdir('.') if not p.startswith('.')))

## Experiment A — Fine-tuning stress test  ⭐ (the main result)
LoRA-fine-tunes Qwen2.5-1.5B **hard** on synthetic internal records to induce memorization (30 epochs, rank-32 LoRA on attention+MLP), then audits **base vs fine-tuned**. A fast *canary* check prints within seconds whether memorization took.

Expected: base ≈ Low, fine-tuned = **High / 淘汰 REJECTED**.

**Expect roughly 40–55 minutes.** Keep the tab open.

In [ ]:
!python finetune_stress.py --model Qwen/Qwen2.5-1.5B --persons 40 --epochs 30

### Download Experiment A results
Produces `results_ft.zip` — **send this back.**

In [ ]:
import shutil
shutil.make_archive('results_ft', 'zip', 'results_ft')
from google.colab import files
files.download('results_ft.zip')

### (Optional) Show the capacity-scaling effect
Repeat the stress test on 3B — fine-tuned 3B is expected to leak more than 1.5B, demonstrating Carlini et al.'s capacity law. Adds ~60–90 min. Remove the `#` to run.

In [ ]:
# !python finetune_stress.py --model Qwen/Qwen2.5-3B --persons 40 --epochs 30 --out-dir results_ft_3b
# import shutil; shutil.make_archive('results_ft_3b','zip','results_ft_3b')
# from google.colab import files; files.download('results_ft_3b.zip')

---
## Experiment B (optional) — base null-floor across scales
Audits base weights only (no fine-tuning). Expected ≈ 0 MER — the honest lower bound. Pilot (1.5B+3B, 350 entries) ≈ 20–30 min; `--full` (1.5B/3B/7B, 1000 entries) ≈ 1.5–3 h. If you already ran the pilot, you can skip this.

In [ ]:
!python run_gpu.py --models Qwen/Qwen2.5-1.5B Qwen/Qwen2.5-3B --n-entries 350
# !python run_gpu.py --full            # full paper config, all three scales
# !python run_gpu.py --full --batch-size 8   # if you hit CUDA out of memory

In [ ]:
import shutil
shutil.make_archive('results_gpu', 'zip', 'results_gpu')
from google.colab import files
files.download('results_gpu.zip')

---
All PII is synthetic and fictitious (national-standard checksums, verified web-absent). No real personal data is used; fine-tuning is on synthetic records only.